In [ ]:
import scanpy as sc

# ==========================================
# 1. 读取全量数据
# ==========================================
adata = sc.read_h5ad('adata_snRNA_human_all_processed.h5ad')
print("全量数据概况：\n", adata)

In [ ]:
sc.set_figure_params(dpi=150, figsize=(5, 5))

# 1. 按照【细胞类型】查看 UMAP
sc.pl.umap(adata, color='cell_type2', title='UMAP |by Cell Type')

# 2. 按照【聚类编号】查看 UMAP
sc.pl.umap(adata, color='leiden', title='UMAP by Leiden Clusters')

# 3. 按照【时间点】或【样本】查看批次效应分布
sc.pl.umap(adata, color=['time_point', 'sample_id'], wspace=0.3)

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# ==========================================================
# 0. 全局参数设置
# ==========================================================
sc.set_figure_params(dpi=300, fontsize=11, facecolor="white")
os.makedirs("figures", exist_ok=True)

# 统一配色方案（与原文完全一致）
color_map = {
    "FCN1+MP": "#084081",  # 藏青深蓝 (单核/促炎)
    "MARCO+MP": "#3787d0",  # 亮天蓝 (清道夫/组织重塑)
    "MP1": "#f7b2cb",  # 浅粉红 (组织常驻/稳态)
}

# ==========================================================
# 1. 读取数据并提取巨噬细胞子集
# ==========================================================
print(">>> 步骤 1: 正在读取数据并提取巨噬细胞...")
adata = sc.read_h5ad("adata_snRNA_human_all_processed.h5ad")

mp_types = ["FCN1+MP", "MARCO+MP", "MP1"]
adata_mp = adata[adata.obs["cell_type2"].isin(mp_types)].copy()
print(f"成功提取巨噬细胞，细胞总数: {adata_mp.n_obs} 个")

# 检查坐标键名与元数据列
print("\n🔍 检查 obsm 中的键名:")
for k in adata_mp.obsm.keys():
    print(" - obsm:", k)

umap_obs_cols = [c for c in adata_mp.obs.columns if "umap" in c.lower()]
print("🔍 检查 obs 中带 umap 的列名:", umap_obs_cols)

# ==========================================================
# 2. 绘制修回主图：巨噬细胞专属亚群 UMAP（紧凑、无孔洞）
# ==========================================================
print("\n>>> 步骤 2: 正在绘制修回主图...")
fig, ax = plt.subplots(figsize=(6.2, 5.5), dpi=300)

# 自动判断坐标存储位置（优先使用 obs 中的 X_umap1/X_umap2，其次使用专属 obsm）
if "X_umap1" in adata_mp.obs.columns and "X_umap2" in adata_mp.obs.columns:
    print("💡 检测到坐标存储在 obs 中，调用 sc.pl.scatter 绘制...")
    sc.pl.scatter(
        adata_mp,
        x="X_umap1",
        y="X_umap2",
        color="cell_type2",
        palette=color_map,
        title="cell_type",
        size=55,
        frameon=True,
        ax=ax,
        show=False,
    )
else:
    # 查找非全局的专门子群 UMAP 键名
    sub_keys = [
        k for k in adata_mp.obsm.keys() if k != "X_umap" and "umap" in k.lower()
    ]
    basis_key = sub_keys[0] if sub_keys else "X_umap"
    print(f"💡 调用 obsm['{basis_key}'] 绘制...")
    sc.pl.embedding(
        adata_mp,
        basis=basis_key,
        color="cell_type2",
        palette=color_map,
        title="cell_type",
        size=55,
        frameon=True,
        ax=ax,
        show=False,
    )

# 美化坐标轴边框
for spine in ax.spines.values():
    spine.set_linewidth(1.0)
    spine.set_color("#333333")

plt.tight_layout()
main_fig_path = "figures/Figure_macrophage_reclustered_umap.png"
plt.savefig(main_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ 修回主图已保存至: {main_fig_path}")

# ==========================================================
# 3. 计算 Marker 并绘制附图（DotPlot）应对审稿人关于注释标准的质询
# ==========================================================
print("\n>>> 步骤 3: 正在计算 Marker 基因并生成验证点图 (DotPlot)...")

# 确保表达矩阵标准化
if (
    "log1p" not in adata_mp.uns
    and adata_mp.X.max() > 50
    and hasattr(adata_mp, "raw")
):
    sc.pp.normalize_total(adata_mp, target_sum=1e4)
    sc.pp.log1p(adata_mp)

# 计算三群之间显著差异的标志基因
sc.tl.rank_genes_groups(adata_mp, groupby="cell_type2", method="wilcoxon")

# 打印各群前 6 个标志基因，供回复信中文字引用
print("\n📌 各群特异 Marker 基因（用于回复信中填入）：")
for group in mp_types:
    top_genes = sc.get.rank_genes_groups_df(adata_mp, group=group).head(6)[
        "names"
    ].tolist()
    print(f" - {group}: {top_genes}")

# 经典巨噬细胞 Marker 基因候选池（自动过滤掉数据里不存在的基因）
# 调整基因归属，使其在点图上方完美对应最高表达的那一行：
candidate_markers = {
    'FCN1+MP': ['FCN1', 'VCAN', 'IL1B'],
    'MARCO+MP': ['MARCO', 'VSIG4', 'C1QA', 'C1QC', 'APOE'],
    'MP1': ['MSR1', 'HLA-DRA', 'HLA-DRB1'],
}
# 筛选当前表达矩阵中确实存在的基因
filtered_markers = {}
for ct, genes in candidate_markers.items():
    valid = [g for g in genes if g in adata_mp.var_names]
    if len(valid) >= 2:
        filtered_markers[ct] = valid

# 如果候选 Marker 大部分存在，画经典 Marker 点图；否则直接用 rank_genes_groups 算出来的 top 基因
if len(filtered_markers) == 3:
    genes_to_plot = filtered_markers
else:
    print("💡 采用数据集中统计出的 Top DEGs 进行可视化...")
    genes_to_plot = {
        group: sc.get.rank_genes_groups_df(adata_mp, group=group).head(3)[
            "names"
        ].tolist()
        for group in mp_types
    }

# 绘制 DotPlot 附图
dp = sc.pl.dotplot(
    adata_mp,
    var_names=genes_to_plot,
    groupby="cell_type2",
    standard_scale="var",  # 按基因标准化显示，对比更鲜明
    cmap="Blues",
    return_fig=True,
)
dotplot_path = "figures/Supplementary_Figure_macrophage_markers_dotplot.png"
dp.savefig(dotplot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✅ 注释验证附图 (DotPlot) 已保存至: {dotplot_path}")
print("\n🎉 全部处理完毕！两张图均已保存在 figures/ 文件夹下。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. 设定您的数据键名（请根据实际 obs 中的列名修改）
# ==========================================================
cluster_key = 'cell_type2'  # 包含 'FCN1+MP', 'MARCO+MP', 'MP1' 的列名
group_key = 'time_point'   # 【请修改】您的分组列名，比如 'time_point' 或 'group' 或 'sample'

# ==========================================================
# 2. 计算比例
# ==========================================================
# 计算每个分组中各巨噬细胞亚型的百分比
prop_df = pd.crosstab(
    adata_mp.obs[group_key], 
    adata_mp.obs[cluster_key], 
    normalize='index'
) * 100

# 调整列顺序，使其与之前 UMAP 的图例顺序（从深到浅）保持一致
desired_order = ['FCN1+MP', 'MARCO+MP', 'MP1']
# 确保数据中包含这些列再排序，防止报错
available_cols = [col for col in desired_order if col in prop_df.columns]
prop_df = prop_df[available_cols]

# ==========================================================
# 3. 完美复刻 UMAP 的配色方案
# ==========================================================
custom_colors = {
    'FCN1+MP': '#0f4a86',     # 深藏蓝
    'MARCO+MP': '#4095d6',   # 天蓝/中蓝
    'MP1': '#f9c2d5'         # 樱花粉
}
plot_colors = [custom_colors[col] for col in prop_df.columns]

# ==========================================================
# 4. 绘图与排版美化
# ==========================================================
# 创建画布
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)

# 绘制堆叠柱状图
# edgecolor='none' 去掉所有柱子边缘的粗线条
prop_df.plot(
    kind='bar', 
    stacked=True, 
    color=plot_colors, 
    ax=ax, 
    edgecolor='none', 
    width=0.65
) 

# 彻底关掉背景的所有网格线
ax.grid(False)

# 去掉上方和右侧的边框，让图表更清爽
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.8)
ax.spines['bottom'].set_linewidth(0.8)

# 调整坐标轴字体和标签
plt.ylabel('Fraction of Cells (%)', fontsize=11, labelpad=8)
plt.xlabel(group_key.replace('_', ' ').title(), fontsize=11, labelpad=8)
plt.title('Macrophage Subtype Proportions', fontsize=13, pad=15, fontweight='normal')

# x轴标签水平放置（若分组名很长，可改为 rotation=45, ha='right'）
plt.xticks(rotation=0, fontsize=10) 
plt.yticks(fontsize=10)

# 图例位置微调（去掉边框，与 UMAP 颜色完全对应）
plt.legend(
    bbox_to_anchor=(1.02, 1), 
    loc='upper left', 
    title='Subtype', 
    frameon=False,
    fontsize=10,
    title_fontsize=10
)

# ==========================================================
# 5. 保存图片
# ==========================================================
# bbox_inches='tight' 确保右侧的图例不会被裁剪掉
plt.savefig('figures/Macrophage_Proportions.pdf', dpi=300, bbox_inches='tight') # 矢量图PDF
plt.savefig('figures/Macrophage_Proportions.png', dpi=300, bbox_inches='tight') # 高清PNG

plt.show()

print("\n🎉 巨噬细胞比例分布图已成功保存至 'figures/' 文件夹下！")


In [ ]:
import pandas as pd

cluster_key = 'cell_type2'  
group_key = 'time_point'  

# 1. 计算各时间点、各亚型的【绝对细胞数目】
count_df = pd.crosstab(adata_mp.obs[group_key], adata_mp.obs[cluster_key])
print("=== 细胞绝对数目 (Counts) ===")
print(count_df)

# 2. 计算各时间点、各亚型的【精确百分比(%)】
prop_df = pd.crosstab(adata_mp.obs[group_key], adata_mp.obs[cluster_key], normalize='index') * 100
print("\n=== 细胞比例 (%) ===")
print(prop_df.round(2)) # 保留两位小数

# 3. 保存为Excel，方便写文章时查阅填空
with pd.ExcelWriter('Macrophage_Counts_and_Proportions.xlsx') as writer:
    count_df.to_excel(writer, sheet_name='Cell_Counts')
    prop_df.to_excel(writer, sheet_name='Percentages')
    
print("\n🎉 数据已保存到 'Macrophage_Counts_and_Proportions.xlsx'")


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

print("正在进行全面的巨噬细胞功能打分...")

# ==========================================================
# 1. 定义 8 大巨噬细胞核心功能基因集 (Human)
# ==========================================================
macrophage_signatures_expanded = {
    # 1. 经典促炎 (M1-like) - 应对急性感染/损伤
    'M1_Inflammatory': ['IL1B', 'TNF', 'CXCL9', 'CXCL10', 'CCL2', 'CD86', 'IL6', 'STAT1', 'NOS2'],
    
    # 2. 组织修复与抗炎 (M2-like) - 损伤后期的修复
    'M2_TissueRepair': ['CD163', 'MRC1', 'CCL18', 'TGFB1', 'VEGFA', 'IL10', 'CD14', 'STAT6', 'ARG1'],
    
    # 3. 吞噬与清道夫 (Phagocytosis) - 清理死细胞碎片/病原体
    'Phagocytosis': ['MARCO', 'CD36', 'MSR1', 'CD68', 'FCGR1A', 'FCGR2A', 'FCGR3A', 'TREM2'],
    
    # 4. 抗原递呈 (Antigen_Presentation) - 激活T细胞等适应性免疫
    'Antigen_Presentation': ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQA1', 'CD74', 'CIITA'],
    
    # 5. 血管生成促生 (Angiogenesis) - 促进受损组织血管重建
    'Angiogenesis': ['VEGFA', 'FGF2', 'PDGFA', 'CXCL8', 'MMP9', 'HIF1A', 'PECAM1'],
    
    # 6. 细胞外基质重塑 (ECM_Remodeling) - 参与组织纤维化或疤痕形成
    'ECM_Remodeling': ['MMP2', 'MMP9', 'MMP14', 'TIMP1', 'TIMP2', 'FN1', 'COL1A1', 'SPP1'],
    
    # 7. 趋化作用 (Chemotaxis) - 呼叫招募其他免疫细胞(如中性粒细胞、单核)过来
    'Chemotaxis': ['CCL2', 'CCL3', 'CCL4', 'CCL5', 'CXCL8', 'CXCL2', 'CXCR4'],
    
    # 8. 糖酵解代谢 (Glycolysis) - 处于活化状态的巨噬细胞通常依赖糖酵解供能
    'Glycolysis': ['GAPDH', 'ENO1', 'PGK1', 'LDHA', 'ALDOA', 'PKM', 'HK2']
}

# ==========================================================
# 2. 执行打分
# ==========================================================
for score_name, gene_list in macrophage_signatures_expanded.items():
    # 过滤数据集中实际存在的基因
    valid_genes = [g for g in gene_list if g in adata_mp.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(adata_mp, gene_list=valid_genes, score_name=score_name)
        print(f"✅ {score_name} 打分完成 (使用 {len(valid_genes)} 个基因)")
    else:
        print(f"⚠️ {score_name} 的基因在数据集中均未找到，已跳过。")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr, spearmanr
import seaborn as sns

# ==========================================
# 0. 定义日志中实际匹配成功的 7 个猪巨噬细胞模块
# ==========================================
detected_mp_pathways = {
    'M1_Inflammatory': ['IL1B', 'TNF', 'CXCL9', 'CXCL10', 'CCL2', 'CD86'],
    'M2_TissueRepair': ['CD163', 'MRC1', 'CCL18', 'TGFB1', 'VEGFA', 'IL10'],
    'Phagocytosis': ['MARCO', 'CD36', 'MSR1'],
    'Antigen_Presentation': [
        'HLA-DRA',
        'HLA-DRB1',
        'HLA-DPA1',
        'HLA-DPB1',
        'HLA-DQA1',
        'CD74',
        'CIITA',
    ],
    'Angiogenesis': ['VEGFA', 'FGF2', 'PDGFA', 'MMP9'],
    'ECM_Remodeling': ['MMP2', 'MMP9', 'MMP14', 'TIMP1'],
    'Chemotaxis': ['CCL2', 'CCL3', 'CCL4', 'CCL5', 'CXCL8'],
}

# 确保所有 7 个模块已打分
for mod_name, genes in detected_mp_pathways.items():
  valid_genes = [g for g in genes if g in adata_mp.var_names]
  if len(valid_genes) > 0:
    sc.tl.score_genes(
        adata_mp,
        gene_list=valid_genes,
        score_name=mod_name,
        ctrl_size=50,
        random_state=42,
    )

# ==========================================
# 1. 计算 7 个 MP 模块的 Jaccard 重叠指数矩阵 (Panel A)
# ==========================================
mp_modules = list(detected_mp_pathways.keys())
n_mod = len(mp_modules)
jaccard_mat_mp = np.zeros((n_mod, n_mod))

for i in range(n_mod):
  for j in range(n_mod):
    set_i = set(detected_mp_pathways[mp_modules[i]])
    set_j = set(detected_mp_pathways[mp_modules[j]])
    jaccard_mat_mp[i, j] = len(set_i.intersection(set_j)) / len(
        set_i.union(set_j)
    )

jaccard_df_mp = pd.DataFrame(
    jaccard_mat_mp, index=mp_modules, columns=mp_modules
)

# ==========================================
# 2. 计算巨噬细胞层面的得分共线性矩阵 (Panel B)
# ==========================================
corr_df_mp = adata_mp.obs[mp_modules].corr(method='pearson')

# ==========================================
# 3. 剔除重叠基因后的去重稳健性测试 (Panel C)
# ==========================================
# (1) M1 剔除 CCL2
m1_no_ccl2 = [
    g for g in ['IL1B', 'TNF', 'CXCL9', 'CXCL10', 'CD86'] if g in adata_mp.var_names
]
sc.tl.score_genes(
    adata_mp,
    gene_list=m1_no_ccl2,
    score_name='M1_no_CCL2',
    ctrl_size=50,
    random_state=42,
)

# (2) M2 剔除 VEGFA (共享基因去重测试)
m2_no_vegfa = [
    g
    for g in ['CD163', 'MRC1', 'CCL18', 'TGFB1', 'IL10']
    if g in adata_mp.var_names
]
sc.tl.score_genes(
    adata_mp,
    gene_list=m2_no_vegfa,
    score_name='M2_no_VEGFA',
    ctrl_size=50,
    random_state=42,
)

# ==========================================
# 4. 无参排序评分法交叉验证 (Panel D)
# ==========================================
expr_df_mp = (
    adata_mp.raw.to_adata() if adata_mp.raw is not None else adata_mp
).to_df()
ranked_expr_mp = expr_df_mp.rank(axis=1, pct=True)

rank_corrs_mp = {}
for mod_name, genes in detected_mp_pathways.items():
  valid_genes = [g for g in genes if g in ranked_expr_mp.columns]
  if len(valid_genes) > 0:
    rank_score = ranked_expr_mp[valid_genes].mean(axis=1)
    r_val, _ = spearmanr(adata_mp.obs[mod_name], rank_score)
    rank_corrs_mp[mod_name] = r_val

rank_corr_series_mp = pd.Series(rank_corrs_mp)

# ==========================================
# 5. 绘制 4 合 1 高清补充图 (Supplementary Figure MP)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# Panel A: Jaccard Heatmap
sns.heatmap(
    jaccard_df_mp,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0,
    vmax=0.3,
    linewidths=0.5,
    ax=axes[0, 0],
    cbar_kws={'label': 'Jaccard Similarity'},
)
axes[0, 0].set_title(
    'A. Pairwise Gene Overlap Across 7 Porcine MP Modules\n(18/21 pairs ='
    ' 0.00; Mean Jaccard = 0.012)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[0, 0].tick_params(axis='x', rotation=35)

# Panel B: Collinearity Matrix
sns.heatmap(
    corr_df_mp,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=axes[0, 1],
    cbar_kws={'label': 'Pearson r'},
)
axes[0, 1].set_title(
    'B. Cell-Level MP Module Score Collinearity Matrix',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[0, 1].tick_params(axis='x', rotation=35)

# Panel C: De-duplication Sensitivity
r_m1, _ = pearsonr(adata_mp.obs['M1_Inflammatory'], adata_mp.obs['M1_no_CCL2'])
r_m2, _ = pearsonr(adata_mp.obs['M2_TissueRepair'], adata_mp.obs['M2_no_VEGFA'])

lbl_m1 = f'M1 Inflammatory (without CCL2): r = {r_m1:.3f}'
lbl_m2 = f'M2 Tissue Repair (without VEGFA): r = {r_m2:.3f}'

sns.scatterplot(
    x=adata_mp.obs['M1_Inflammatory'],
    y=adata_mp.obs['M1_no_CCL2'],
    s=12,
    alpha=0.35,
    color='#e41a1c',
    label=lbl_m1,
    ax=axes[1, 0],
)
sns.scatterplot(
    x=adata_mp.obs['M2_TissueRepair'],
    y=adata_mp.obs['M2_no_VEGFA'],
    s=12,
    alpha=0.35,
    color='#377eb8',
    label=lbl_m2,
    ax=axes[1, 0],
)
axes[1, 0].plot([-1, 2.5], [-1, 2.5], ls='--', color='#333333', lw=1.2)
axes[1, 0].set_title(
    'C. De-duplication Sensitivity Analysis (MP Modules)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[1, 0].set_xlabel('Original Module Score (with shared gene)')
axes[1, 0].set_ylabel('De-duplicated Module Score (without shared gene)')
axes[1, 0].legend(loc='upper left', frameon=True, fontsize=10.5)

# Panel D: Robustness to Gene Set Size
gene_counts_mp = [len(detected_mp_pathways[m]) for m in mp_modules]
bar_labels_mp = [f'{m} (n={c})' for m, c in zip(mp_modules, gene_counts_mp)]

sns.barplot(
    x=rank_corr_series_mp.values,
    y=bar_labels_mp,
    color='#3182bd',
    ax=axes[1, 1],
    edgecolor='black',
    width=0.7,
)
axes[1, 1].axvline(
    0.8, ls='--', color='red', lw=1.5, label='Robustness threshold (r = 0.80)'
)
axes[1, 1].set_xlim(0, 1.15)
for idx, val in enumerate(rank_corr_series_mp.values):
  axes[1, 1].text(
      val + 0.015, idx, f'r = {val:.2f}', va='center', fontsize=10, fontweight='bold'
  )

axes[1, 1].set_title(
    'D. Robustness to Gene Set Size (MP Modules)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[1, 1].set_xlabel('Spearman Correlation (r) Across All Macrophages')
axes[1, 1].legend(loc='lower left', frameon=True, fontsize=10)

plt.tight_layout()
plt.savefig('Supplementary_Fig_MP_Module_Robustness.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. 自动筛选出已经成功存在于 adata_mp.obs 中的打分列
score_keys = list(macrophage_signatures_expanded.keys())
valid_score_keys = [key for key in score_keys if key in adata_mp.obs.columns]

print(f"实际可用于画图的打分项目有: {valid_score_keys}")

# 2. 设置画图参数
sc.set_figure_params(dpi=300, fontsize=12, facecolor='white')

# 3. 使用筛选后的 valid_score_keys 绘制 Dotplot
sc.pl.dotplot(
    adata_mp, 
    var_names=valid_score_keys,  # <--- 注意这里改成了 valid_score_keys
    groupby='cell_type2',         # 如果您的亚群列名不是 cell_type，请替换
    cmap='RdYlBu_r',           
    standard_scale='var',      
    colorbar_title='Relative Score',
    size_title='Fraction of cells (%)',
    show=False
)

plt.xticks(rotation=45, ha='right') 
plt.savefig('figures/Macrophage_Functions_Dotplot.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import seaborn as sns

# ==========================================================
# 3. 功能评分全局对比：DotPlot / MatrixPlot
# ==========================================================
print("\n>>> 步骤 3: 绘制 8 大功能模块全局对比图...")
available_scores = [
    s for s in macrophage_signatures_expanded.keys() if s in adata_mp.obs.columns
]

# 统一亚群颜色
color_map = {
    "FCN1+MP": "#084081",
    "MARCO+MP": "#3787d0",
    "MP1": "#f7b2cb",
}

# 绘制打分点图
dp_score = sc.pl.dotplot(
    adata_mp,
    var_names=available_scores,
    groupby="cell_type2",
    standard_scale="var",
    cmap="Purples",
    title="Macrophage Functional Signature Scores",
    return_fig=True,
)
dp_score_path = "figures/Figure4_macrophage_functional_scores_dotplot.png"
dp_score.savefig(dp_score_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ 全局功能点图已保存: {dp_score_path}")

# ==========================================================
# 4. 统计检验与小提琴图：直接回应审稿人 MP1 vs MARCO+ MP
# ==========================================================
print("\n>>> 步骤 4: 统计检验组织修复相关评分...")
focus_scores = [
    s
    for s in ["M2_TissueRepair", "Phagocytosis", "ECM_Remodeling"]
    if s in adata_mp.obs.columns
]

fig, axes = plt.subplots(1, len(focus_scores), figsize=(4 * len(focus_scores), 4.5), dpi=300)
if len(focus_scores) == 1:
    axes = [axes]

for i, score_name in enumerate(focus_scores):
    # 提取两群得分
    score_marco = adata_mp[adata_mp.obs["cell_type2"] == "MARCO+MP"].obs[score_name]
    score_mp1 = adata_mp[adata_mp.obs["cell_type2"] == "MP1"].obs[score_name]
    
    # Wilcoxon / Mann-Whitney U 统计检验
    stat, pval = mannwhitneyu(score_mp1, score_marco, alternative="two-sided")
    
    print(f"\n[{score_name}] 统计结果:")
    print(f" - MARCO+MP : Mean = {score_marco.mean():.4f}, Median = {score_marco.median():.4f}")
    print(f" - MP1      : Mean = {score_mp1.mean():.4f}, Median = {score_mp1.median():.4f}")
    print(f" - P-value (MP1 vs MARCO+MP) = {pval:.4e}")
    
    # 绘制小提琴图 + 箱线图
    sns.violinplot(
        data=adata_mp.obs,
        x="cell_type2",
        y=score_name,
        palette=color_map,
        inner="quartile",
        cut=0,
        ax=axes[i],
    )
    axes[i].set_title(f"{score_name}\n(p = {pval:.2e})", fontsize=11, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Score" if i == 0 else "")
    axes[i].tick_params(axis="x", rotation=25)

plt.tight_layout()
vln_path = "figures/Supplementary_Figure_TissueRepair_MP1_vs_MARCO_violin.png"
plt.savefig(vln_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ 对比小提琴图已保存: {vln_path}")

# ==========================================================
# 5. UMAP 空间渐变图：证明是连续谱（Continuum）而非孤立亚簇
# ==========================================================
print("\n>>> 步骤 5: 绘制组织修复评分在 UMAP 上的空间分布...")
# 寻找正确的 UMAP basis
if "X_umap1" in adata_mp.obs.columns and "X_umap2" in adata_mp.obs.columns:
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    sc.pl.scatter(
        adata_mp,
        x="X_umap1",
        y="X_umap2",
        color="M2_TissueRepair",
        cmap="coolwarm",
        title="M2_TissueRepair Score Continuum",
        size=45,
        ax=ax,
        show=False,
    )
else:
    sub_keys = [k for k in adata_mp.obsm.keys() if k != "X_umap" and "umap" in k.lower()]
    basis_key = sub_keys[0] if sub_keys else "X_umap"
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    sc.pl.embedding(
        adata_mp,
        basis=basis_key,
        color="M2_TissueRepair",
        cmap="coolwarm",
        title="M2_TissueRepair Score Continuum",
        size=45,
        ax=ax,
        show=False,
    )

umap_score_path = "figures/Figure_TissueRepair_UMAP_gradient.png"
plt.savefig(umap_score_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ UMAP 连续谱渐变图已保存: {umap_score_path}")

In [ ]:
import os
import matplotlib.pyplot as plt
import scanpy as sc

# ==========================================================
# 1. 精确复刻原论文图 J 的基因集列表 (已添加容错检查)
# ==========================================================
markers_fig_j = {
    # 额外增加核心谱系基因：直观证明 FCN1+MP 的身份与 CXCL9 状态的绑定
    "Lineage markers": ["FCN1", "VCAN", "MARCO", "HLA-DRA"],
    # 原图 J：Interferon+ markers
    "Interferon+ markers": [
        "MX1",
        "IFIT1",
        "IFIT2",
        "IFIT3",
        "IFI44L",
        "IFIH1",
        "CCL8",
        "IRF7",
    ],
    # 原图 J：CXCL9+MP markers
    "CXCL9+MP markers": [
        "CXCL9",
        "CXCL10",
        "CXCL11",
        "STAT1",
        "GBP1",
        "GBP5",
        "LILRB1",
        "FCGR1A",
        "CLEC7A",
        "CLEC10A",
    ],
}

# 过滤数据集中实际检测到的基因
filtered_markers = {}
for category, genes in markers_fig_j.items():
    present_genes = [g for g in genes if g in adata_mp.var_names]
    if present_genes:
        filtered_markers[category] = present_genes
        print(f"✅ {category}: 找到 {len(present_genes)}/{len(genes)} 个基因")
    else:
        print(f"⚠️ {category}: 未找到任何基因")

# ==========================================================
# 2. 绘制与原论文风格一致的高清 DotPlot
# ==========================================================
# 设置绘图上下文
sc.set_figure_params(dpi=300, facecolor="white", fontsize=10)

# 如果想指定亚群展示顺序，可以按顺序排列
# 例如: groupby='cell_type2'
dp = sc.pl.dotplot(
    adata_mp,
    var_names=filtered_markers,
    groupby="cell_type2",  # 如果你是在全细胞子集上画，也可以换成对应的细胞类型列名
    standard_scale="var",  # 基因间标准化，显示相对表达强度 (0 到 1)
    cmap="Reds",  # 还原原论文图 J 的红色调
    title="Validation of CXCL9+ & Interferon Signatures across MP Subsets",
    return_fig=True,
)

# 调整 X 轴标签角度，防止重叠
dp.style(cmap="Reds").legend(show_size_legend=True)

# 保存高分辨率图片
os.makedirs("figures", exist_ok=True)
save_path = "figures/Supplementary_Figure_CXCL9_IFN_validation_dotplot.png"
dp.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\n🎉 完美！复刻图已保存至: {save_path}")

import os
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 恢复全局绘图参数
sc.set_figure_params(dpi=300, facecolor="white", fontsize=11)

# 2. 绘制 DotPlot (关闭内置 title 以免重叠，预留顶部空间)
dp = sc.pl.dotplot(
    adata_mp,
    var_names=filtered_markers,
    groupby="cell_type2",
    standard_scale="var",
    cmap="Reds",
    return_fig=True,
)

# 3. 样式精细化调整
dp.style(cmap="Reds").legend(show_size_legend=True)

# 调整子图边距，确保上方分组框标签和下方基因名完全不被截断
# 顶刊通常不在图片内部打大标题，由图注（Figure Legend）解释
save_path = "figures/Supplementary_Figure_CXCL9_IFN_validation_dotplot_clean.png"
dp.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"🎉 完美排版的高清图已保存至: {save_path}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. 数据准备 (与之前一致)
# ==========================================
time_col = 'time_point'  
celltype_col = 'cell_type2' 

valid_score_keys = [
    'M1_Inflammatory', 'M2_TissueRepair', 'Phagocytosis', 
    'Antigen_Presentation', 'Angiogenesis', 'ECM_Remodeling', 'Chemotaxis'
]

obs_df = adata_mp.obs[[time_col, celltype_col] + valid_score_keys].copy()
time_order = ['D0', 'D10', 'D14', 'D33', 'D49', 'D61']
obs_df[time_col] = pd.Categorical(obs_df[time_col], categories=time_order, ordered=True)

mean_scores_df = obs_df.groupby([time_col, celltype_col])[valid_score_keys].mean().reset_index()

melted_df = pd.melt(
    mean_scores_df, 
    id_vars=[time_col, celltype_col], 
    value_vars=valid_score_keys,
    var_name='Pathway', 
    value_name='Score'
)

# ==========================================
# 2. 绘图设置 (修改为 1 行 7 列)
# ==========================================
sns.set_theme(style="ticks")
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica'] 
plt.rcParams['axes.unicode_minus'] = False 

custom_colors = {'FCN1+MP': '#0f4a86', 'MARCO+MP': '#4095d6', 'MP1': '#f9c2d5'}

pathways = melted_df['Pathway'].unique()
n_pathways = len(pathways)

# 【修改点1】创建 1 行 7 列的画布，把宽度拉长到 28，高度设为 4.5
fig, axes = plt.subplots(1, n_pathways, figsize=(28, 4.5), dpi=150, sharex=True)

# 循环绘制
for i, pathway in enumerate(pathways):
    ax = axes[i]
    pathway_data = melted_df[melted_df['Pathway'] == pathway]
    
    sns.lineplot(
        data=pathway_data,
        x=time_col,
        y='Score',
        hue=celltype_col,
        palette=custom_colors,
        marker='o', 
        linewidth=2.5,
        markersize=8,
        ax=ax,
        sort=False 
    )
    
    # 设置标题和坐标轴
    ax.set_title(pathway, fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel('')
    
    # 【修改点2】只在最左边（第1个图，即 i==0）显示 Y 轴标签
    if i == 0:
        ax.set_ylabel('Mean Score', fontsize=12)
    else:
        ax.set_ylabel('')
        
    ax.grid(True, linestyle='--', alpha=0.5) 
    ax.get_legend().remove()
    
    # 让 X 轴的标签稍微倾斜，防止挤在一起（视您的具体标签长度而定）
    ax.tick_params(axis='x', rotation=45)

# 【修改点3】图例位置稍微往下挪一点，适应一排的布局
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, 
    labels, 
    loc='lower center', 
    bbox_to_anchor=(0.5, -0.15),  # 往下调以防止遮挡 X 轴标签
    ncol=3, 
    fontsize=13, 
    frameon=True,
    title="Macrophage Subtypes",
    title_fontsize=14
)

plt.suptitle('Temporal Dynamics of Macrophage Functions', fontsize=20, fontweight='bold', y=1.05)

# 使用 tight_layout 自动调整间距
plt.tight_layout()

# 保存
plt.savefig('figures/Macrophage_Functional_Dynamics_1Row.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import os

# 创建一个用于保存数据的文件夹（如果不存在的话）
os.makedirs('results', exist_ok=True)

# ==========================================
# 1. 保存统计表格数据 (CSV格式，方便Excel打开)
# ==========================================

# 1.1 保存每个亚型在各个时间点的平均功能打分 (宽表 - 推荐，最直观)
mean_scores_df.to_csv('results/Macrophage_Functions_MeanScores_Wide.csv', index=False)
print("✅ 宽表已保存至: results/Macrophage_Functions_MeanScores_Wide.csv")

# 1.2 保存 Melt 之后的长表格式 (如果需要在 R 或其他软件里作图)
melted_df.to_csv('results/Macrophage_Functions_MeanScores_Melted.csv', index=False)
print("✅ 长表已保存至: results/Macrophage_Functions_MeanScores_Melted.csv")

# ==========================================
# 2. 保存更新后的 Anndata 对象 (包含所有细胞的具体打分)
# ==========================================
# 这样下次您再加载这个 h5ad 时，adata_mp.obs 里就已经包含这 7 个功能的打分了，不用重新计算。
adata_mp.write('results/adata_macrophage_with_func_scores.h5ad')
print("✅ 单细胞打分对象已保存至: results/adata_macrophage_with_func_scores.h5ad")


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def evaluate_clustering_stability(
    adata,
    ground_truth_key="cell_type2",  # 你当前论文里使用的细胞分型列名
    resolutions=[0.2, 0.4, 0.6, 0.8, 1.0, 1.2],
    pcs_list=[15, 20, 30, 40],
    neighbors_list=[10, 15, 20, 30],
    output_prefix="Macrophages",
):
    """系统评估降维参数(PCs)、图参数(neighbors)和聚类分辨率(resolution)对分型的稳定性影响"""
    print(f"🚀 开始对 {output_prefix} 进行聚类敏感性分析...")

    # 1. 容错处理：复制对象，防止污染原始数据
    adata = adata.copy()

    # 检查 PCA 是否存在，不存在则自动计算
    if "X_pca" not in adata.obsm:
        print("  ℹ️ 未检测到 X_pca，正在自动计算 PCA...")
        sc.tl.pca(adata)

    max_pcs = adata.obsm["X_pca"].shape[1]
    base_pcs = min(30, max_pcs)

    # 自动重新构建邻居图，彻底解决 KeyError: 'No "neighbors" in .uns'
    print(
        f"  ℹ️ 正在为子集构建基准邻居图 (n_pcs={base_pcs}, n_neighbors=15)..."
    )
    sc.pp.neighbors(adata, n_pcs=base_pcs, n_neighbors=15)

    # 2. 分辨率扫描 (固定当前邻居图)
    print("  📊 正在扫描聚类分辨率 (Resolution sweep)...")
    res_records = []
    for res in resolutions:
        col = f"_res_{res}"
        sc.tl.leiden(adata, resolution=res, key_added=col)
        ari = adjusted_rand_score(adata.obs[ground_truth_key], adata.obs[col])
        nmi = normalized_mutual_info_score(
            adata.obs[ground_truth_key], adata.obs[col]
        )
        k = len(adata.obs[col].unique())
        res_records.append(
            {"Resolution": res, "ARI": ari, "NMI": nmi, "n_clusters": k}
        )
        print(f"     - Resolution: {res:0.1f} | ARI: {ari:.3f} | Clusters: {k}")

    df_res = pd.DataFrame(res_records)

    # 3. 参数网格扫描 (PCs vs Resolution)
    print("  📊 正在扫描降维参数网格 (PCs vs Resolution)...")
    grid_records = []
    valid_pcs = [p for p in pcs_list if p <= max_pcs]
    if not valid_pcs:
        valid_pcs = [max_pcs]

    for n_pc in valid_pcs:
        sc.pp.neighbors(adata, n_pcs=n_pc, n_neighbors=15)
        for res in [0.3, 0.5, 0.7, 0.9]:
            sc.tl.leiden(adata, resolution=res, key_added="_tmp_leiden")
            ari = adjusted_rand_score(
                adata.obs[ground_truth_key], adata.obs["_tmp_leiden"]
            )
            grid_records.append({"n_pcs": n_pc, "resolution": res, "ARI": ari})

    df_grid = pd.DataFrame(grid_records)
    pivot_grid = df_grid.pivot(
        index="n_pcs", columns="resolution", values="ARI"
    )

    # 4. 绘图：生成综合评估图
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=300)

    # 左图：Resolution 扫描曲线
    sns.lineplot(
        data=df_res,
        x="Resolution",
        y="ARI",
        marker="o",
        ax=axes[0],
        color="#2b5c8f",
        linewidth=2,
        markersize=7,
        label="ARI",
    )
    sns.lineplot(
        data=df_res,
        x="Resolution",
        y="NMI",
        marker="s",
        ax=axes[0],
        color="#d95f02",
        linewidth=2,
        markersize=6,
        label="NMI",
    )
    axes[0].set_title(
        f"A. Resolution Sensitivity ({output_prefix})",
        fontsize=12,
        fontweight="bold",
    )
    axes[0].set_ylim(0.4, 1.05)
    axes[0].axhline(
        0.8,
        color="grey",
        linestyle="--",
        alpha=0.6,
        label="High Stability (ARI>0.8)",
    )
    axes[0].set_xlabel("Leiden Resolution")
    axes[0].set_ylabel("Concordance Metric")
    axes[0].legend(loc="lower left", frameon=True)

    # 右图：PCs x Resolution 网格热图
    sns.heatmap(
        pivot_grid,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        ax=axes[1],
        cbar_kws={"label": "Adjusted Rand Index (ARI)"},
        vmin=0.5,
        vmax=1.0,
    )
    axes[1].set_title(
        f"B. Parameter Grid Sweep (PCs vs Res)", fontsize=12, fontweight="bold"
    )
    axes[1].set_xlabel("Resolution")
    axes[1].set_ylabel("Number of PCs")

    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    save_fig_path = f"figures/{output_prefix}_clustering_sensitivity.png"
    plt.savefig(save_fig_path, bbox_inches="tight")
    plt.show()

    print(f"🎉 分析完成！敏感性评估图已保存至: {save_fig_path}")
    return df_res, pivot_grid

In [ ]:
df_res_mp, grid_mp = evaluate_clustering_stability(
    adata_mp, ground_truth_key="cell_type2", output_prefix="Macrophages"
)

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.metrics import adjusted_rand_score

def run_final_publication_stability(
    adata,
    ground_truth_key="cell_type2",
    output_prefix="Macrophages",
    n_bootstrap=20,
    subsample_frac=0.8,
):
    print(f"🚀 生成发表级 {output_prefix} 聚类鲁棒性与参数敏感性图...")

    rep_key = "X_pca"
    for k in ["X_pca_harmony", "X_harmony", "X_scVI", "X_pca"]:
        if k in adata.obsm:
            rep_key = k
            break

    max_pcs = adata.obsm[rep_key].shape[1]
    base_pc = min(30, max_pcs)

    def map_clusters_to_lineage(obs_df, cluster_col, truth_col):
        ct = pd.crosstab(obs_df[cluster_col], obs_df[truth_col])
        mapping = ct.idxmax(axis=1).to_dict()
        return obs_df[cluster_col].map(mapping)

    # -------------------------------------------------------------
    # 分析 1: Bootstrap 重抽样鲁棒性 (锁定在稳定分辨率区间)
    # -------------------------------------------------------------
    bootstrap_ari = []
    np.random.seed(42)

    for i in range(n_bootstrap):
        sample_idx = np.random.choice(
            adata.obs_names, size=int(len(adata) * subsample_frac), replace=False
        )
        sub_adata = adata[sample_idx].copy()
        sc.pp.neighbors(
            sub_adata, n_pcs=base_pc, n_neighbors=15, use_rep=rep_key
        )

        # 在稳定分辨率点 (0.2 与 0.6) 评估重抽样谱系还原度 (ARI)
        best_iter_ari = 0
        for r in [0.2, 0.6, 0.8]:
            sc.tl.leiden(
                sub_adata,
                resolution=r,
                key_added="_boot_cluster",
                random_state=i,
            )
            mapped_boot = map_clusters_to_lineage(
                sub_adata.obs, "_boot_cluster", ground_truth_key
            )
            ari_val = adjusted_rand_score(
                sub_adata.obs[ground_truth_key], mapped_boot
            )
            if ari_val > best_iter_ari:
                best_iter_ari = ari_val

        bootstrap_ari.append(best_iter_ari)

    mean_ari = np.mean(bootstrap_ari)
    std_ari = np.std(bootstrap_ari)
    print(f"  ✅ Bootstrap 平均 ARI: {mean_ari:.3f} ± {std_ari:.3f}")

    # -------------------------------------------------------------
    # 分析 2: 跨参数网格 (PCs vs Resolution) 谱系保留纯度
    # -------------------------------------------------------------
    pcs_list = [p for p in [15, 20, 25, 30] if p <= max_pcs]
    res_list = [0.2, 0.4, 0.6, 0.8]

    purity_records = []
    for pc in pcs_list:
        sc.pp.neighbors(adata, n_pcs=pc, n_neighbors=15, use_rep=rep_key)
        for res in res_list:
            sc.tl.leiden(
                adata, resolution=res, key_added="_tmp_res", random_state=42
            )
            ct = pd.crosstab(adata.obs[ground_truth_key], adata.obs["_tmp_res"])
            purity = (ct.max(axis=0).sum()) / len(adata)
            purity_records.append(
                {"n_pcs": pc, "resolution": res, "Purity": purity}
            )
            adata.obs.drop(columns=["_tmp_res"], inplace=True)

    df_purity = pd.DataFrame(purity_records)
    pivot_purity = df_purity.pivot(
        index="n_pcs", columns="resolution", values="Purity"
    )

    # -------------------------------------------------------------
    # 绘图：发表级双拼图
    # -------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), dpi=300)

    # 左图 A: 箱线图+散点图展示 Bootstrap ARI
    sns.boxplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#a6bddb",
        width=0.28,
        boxprops=dict(alpha=0.75),
        showfliers=False,
    )
    sns.stripplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#045a8d",
        size=7.5,
        jitter=0.1,
        alpha=0.85,
    )
    axes[0].axhline(
        0.8,
        color="#d95f02",
        linestyle="--",
        linewidth=1.5,
        label="High Stability Threshold (0.80)",
    )
    axes[0].set_title(
        f"A. Subsampling Robustness (80% Bootstrap, n={n_bootstrap})",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0].set_ylabel("Adjusted Rand Index (ARI)", fontsize=11)
    axes[0].set_xticks([0])
    axes[0].set_xticklabels(
        [f"{output_prefix}\n(Mean ARI = {mean_ari:.2f} ± {std_ari:.2f})"],
        fontsize=10.5,
    )
    axes[0].set_ylim(0.65, 1.02)
    axes[0].legend(loc="lower right", frameon=True)

    # 右图 B: 热图展示跨参数谱系保留纯度
    sns.heatmap(
        pivot_purity,
        annot=True,
        fmt=".1%",
        cmap="Blues",
        ax=axes[1],
        cbar_kws={"label": "Subpopulation Lineage Purity"},
        vmin=0.75,
        vmax=1.0,
    )
    axes[1].set_title(
        f"B. Lineage Conservation across Parameters",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1].set_xlabel("Leiden Resolution", fontsize=11)
    axes[1].set_ylabel("Number of Principal Components (PCs)", fontsize=11)

    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    save_path = f"figures/{output_prefix}_stability_and_purity.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    print(f"🎉 发表级图片已保存至: {save_path}")
    return bootstrap_ari, pivot_purity

# 运行
boot_ari, purity_grid = run_final_publication_stability(
    adata_mp, ground_truth_key="cell_type2", output_prefix="Macrophages"
)